In [0]:
# Verify the raw files are accessible
files = dbutils.fs.ls("/Volumes/workspace/default/superstore_raw/")
for f in files:
    print(f.name, f.size)

In [0]:
# Define the base path to our raw files
RAW_PATH = "/Volumes/workspace/default/superstore_raw/"

# Read Orders CSV with proper quote handling
orders_df = (spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .option("quote", '"')
    .option("escape", '"')
    .option("multiLine", "true")
    .csv(RAW_PATH + "Orders.csv"))

print(f"Orders rows: {orders_df.count()}")
orders_df.printSchema()

In [0]:
# Read People CSV
people_df = (spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .option("quote", '"')
    .option("escape", '"')
    .option("multiLine", "true")
    .csv(RAW_PATH + "People.csv"))

print(f"People rows: {people_df.count()}")
people_df.printSchema()

In [0]:
# Read Returns CSV
returns_df = (spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .option("quote", '"')
    .option("escape", '"')
    .option("multiLine", "true")
    .csv(RAW_PATH + "Returns.csv"))

print(f"Returns rows: {returns_df.count()}")
returns_df.printSchema()

In [0]:
# Clean column names: replace spaces with underscores
def clean_columns(df):
    for col_name in df.columns:
        new_name = col_name.replace(" ", "_")
        df = df.withColumnRenamed(col_name, new_name)
    return df

In [0]:
# Clean and write Orders as Bronze Delta table
orders_df = clean_columns(orders_df)
orders_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.bronze_orders")
print("bronze_orders table created")

In [0]:
# Clean and write People
people_df = clean_columns(people_df)
people_df.write.format("delta").mode("overwrite").saveAsTable("workspace.default.bronze_people")
print("bronze_people table created")

In [0]:
# Clean and write Returns
returns_df = clean_columns(returns_df)
returns_df.write.format("delta").mode("overwrite").saveAsTable("workspace.default.bronze_returns")
print("bronze_returns table created")

In [0]:
# Read from Bronze and register as temp view
bronze_orders = spark.read.table("workspace.default.bronze_orders")
bronze_orders.createOrReplaceTempView("bronze_orders_view")

# Silver transformation using SQL try_cast
silver_orders = spark.sql("""
    SELECT
        try_cast(Row_ID AS INT)          AS Row_ID,
        Order_ID,
        try_cast(Order_Date AS DATE)     AS Order_Date,
        try_cast(Ship_Date AS DATE)      AS Ship_Date,
        Ship_Mode,
        Customer_ID,
        Customer_Name,
        Segment,
        City,
        State,
        Country,
        CASE 
            WHEN Postal_Code IS NOT NULL 
            THEN regexp_replace(Postal_Code, '\\.0$', '')
            ELSE NULL 
        END                              AS Postal_Code,
        Market,
        Region,
        Product_ID,
        Category,
        `Sub-Category`                   AS Sub_Category,
        Product_Name,
        try_cast(Sales AS DOUBLE)        AS Sales,
        try_cast(Quantity AS INT)        AS Quantity,
        try_cast(Discount AS DOUBLE)     AS Discount,
        try_cast(Profit AS DOUBLE)       AS Profit,
        try_cast(Shipping_Cost AS DOUBLE) AS Shipping_Cost,
        Order_Priority
    FROM bronze_orders_view
    WHERE try_cast(Sales AS DOUBLE) IS NOT NULL
""")

print(f"Silver orders rows: {silver_orders.count()}")
silver_orders.printSchema()

In [0]:
# Write Silver orders as a Delta table
silver_orders.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.silver_orders")
print("silver_orders table created")

In [0]:
# Write rejected rows to quarantine table
bronze_orders.createOrReplaceTempView("bronze_orders_view")

rejected_orders = spark.sql("""
    SELECT * FROM bronze_orders_view
    WHERE try_cast(Sales AS DOUBLE) IS NULL
""")

print(f"Rejected rows: {rejected_orders.count()}")

rejected_orders.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.bronze_rejected")
print("bronze_rejected table created")

In [0]:
# Silver People and Returns - read from Bronze
bronze_people = spark.read.table("workspace.default.bronze_people")
bronze_returns = spark.read.table("workspace.default.bronze_returns")

# People is clean - write directly to Silver
bronze_people.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.silver_people")
print("silver_people table created")

# Returns is clean - write directly to Silver  
bronze_returns.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.silver_returns")
print("silver_returns table created")

In [0]:
# Check schema of each Silver table
print("=== silver_orders schema ===")
spark.table("silver_orders").printSchema()

print("=== silver_people schema ===")
spark.table("silver_people").printSchema()

print("=== silver_returns schema ===")
spark.table("silver_returns").printSchema()

In [0]:
display(spark.table("silver_orders").limit(20))

In [0]:
from pyspark.sql import functions as F

gold_sales_by_region = (
    spark.table("silver_orders")
    .groupBy("Market", "Region")
    .agg(
        F.sum("Sales").alias("Total_Sales"),
        F.sum("Profit").alias("Total_Profit"),
        F.countDistinct("Order_ID").alias("Order_Count")
    )
    .orderBy("Market", F.desc("Total_Sales"))
)

display(gold_sales_by_region)

In [0]:
gold_sales_by_region.write.format("delta").mode("overwrite").saveAsTable("gold_sales_by_region")

In [0]:
from pyspark.sql import functions as F

gold_profit_by_category = (
    spark.table("silver_orders")
    .groupBy("Category", "Sub_Category")
    .agg(
        F.sum("Sales").alias("Total_Sales"),
        F.sum("Profit").alias("Total_Profit"),
        F.countDistinct("Order_ID").alias("Order_Count")
    )
    .withColumn("Profit_Margin_Pct", F.round((F.col("Total_Profit") / F.col("Total_Sales")) * 100, 2))
    .orderBy("Category", F.desc("Total_Profit"))
)

display(gold_profit_by_category)

In [0]:
gold_profit_by_category.write.format("delta").mode("overwrite").saveAsTable("gold_profit_by_category")

In [0]:
# NOTE: Order_ID is not globally unique in this dataset — 37 Order_IDs
# span more than one Market. Joining on Order_ID + Market together
# prevents cross-market return data from being misattributed.
orders_with_returns = (
    spark.table("silver_orders")
    .join(
        spark.table("silver_returns").select("Order_ID", "Market", "Returned"),
        on=["Order_ID", "Market"],
        how="left"
    )
    .withColumn("Is_Returned", F.when(F.col("Returned") == "Yes", 1).otherwise(0))
)

In [0]:
orders_with_returns = (
    spark.table("silver_orders")
    .join(
        spark.table("silver_returns").select("Order_ID", "Market", "Returned"),
        on=["Order_ID", "Market"],
        how="left"
    )
    .withColumn("Is_Returned", F.when(F.col("Returned") == "Yes", 1).otherwise(0))
)

display(orders_with_returns.select("Order_ID", "Market", "Category", "Returned", "Is_Returned").limit(20))

In [0]:
gold_return_rates = (
    orders_with_returns
    .groupBy("Category")
    .agg(
        F.countDistinct("Order_ID", "Market").alias("Total_Orders"),
        F.sum("Is_Returned").alias("Returned_Orders")
    )
    .withColumn("Return_Rate_Pct", F.round((F.col("Returned_Orders") / F.col("Total_Orders")) * 100, 2))
    .orderBy(F.desc("Return_Rate_Pct"))
)

display(gold_return_rates)

In [0]:
gold_return_rates.write.format("delta").mode("overwrite").saveAsTable("gold_return_rates")